# Geldium | Predictive Delinquency Modeling Strategy
### Tata GenAI-Powered Data Analytics · Task 2

**Deliverable:** Conceptual model design and evaluation framework  
**Source:** Geldium exploratory data analysis (Task 1)  
**Status:** Proposed methodology — **models have not been trained or evaluated**

> **Decision recommendation:** Begin with interpretable, regularized **Logistic Regression** as the primary benchmark; evaluate **Decision Tree**, **Random Forest**, and **Gradient Boosting** as challengers. Promote a more complex model only if it provides reliable, meaningful out-of-sample improvement without unacceptable fairness, calibration, or explainability trade-offs.

---
**Notebook guide:** Run only the optional illustrative cells if desired. They do not fit models or generate performance claims. This document is intentionally designed to satisfy a *conceptual modeling* assignment.

## Executive summary

**Business question:** Which customers are at elevated risk of delinquency, and how can Geldium prioritize proportionate, explainable interventions?

**What EDA established:** Of 500 customers, **80 (16%)** were delinquent and **420 (84%)** were not. Individual numeric associations with delinquency were negligible (all reported absolute Pearson correlations below 0.05). Tested categorical associations were weak (reported Cramér’s V below 0.10, with no reported p-value below 0.05). This does **not** establish that multivariable models will fail, but it means no feature should be presented as a proven risk driver.

**Proposed output:** A calibrated probability of delinquency between 0 and 1, a review category derived from a policy-selected threshold, and a concise explanation of relevant model inputs. Probabilities and categories require empirical validation before operational use.

**Success criterion:** Better identification of delinquent accounts than a no-skill baseline on untouched data, with acceptable precision, recall, calibration, stability, fairness, and auditability.

## 1 · Business objective and prediction definition

| Item | Proposed definition |
|---|---|
| Target | `Delinquent_Account` (1 = delinquent, 0 = non-delinquent) |
| Unit of prediction | One customer/account record |
| Output | Estimated probability `P(Delinquent_Account = 1 | available inputs)` |
| Decision use | Prioritize **human-reviewed** outreach and support; not automatic adverse credit decisions |
| Prediction horizon | **To be confirmed**: the dataset does not establish whether the label refers to the next 30, 60, or 90 days |
| Observation cutoff | All predictors must exist **before** the prediction date |

**Critical requirement:** Confirm the target definition, observation date, and prediction horizon with the data owner. If `Month_1`–`Month_6` or `Missed_Payments` were recorded after the target outcome, using them would leak future information. Do not train a production model until timing is verified.

## 2 · EDA evidence used in the design

| Evidence from Task 1 | Modeling implication |
|---|---|
| 500 rows; 80 positives (16%) | Small dataset; stratified evaluation and uncertainty intervals are essential |
| 84% non-delinquent | Always compare against majority-class and prevalence-based baselines; accuracy alone is misleading |
| Missing: Income 39, Loan Balance 29, Credit Score 2 | Train-only imputation; consider missingness indicator; avoid leakage |
| 4 utilization observations above 100% | Validate with business rules; don't discard automatically |
| Numeric correlations with target all \|r\| < 0.05 | No validated standalone numerical driver; consider joint/nonlinear patterns carefully |
| Categorical Cramér’s V < 0.10; none significant at 0.05 | Treat segment differences as exploratory rather than proven effects |
| Payment history: six categorical months | Candidate behavioral features only if available before prediction cutoff |
| Loan Balance missing group: 24.14% delinquent vs 15.50% when present; Fisher p=0.292 | Indicator is exploratory, not a statistically established signal |

**Important distinction:** Features proposed below are based on business plausibility and availability, **not** on demonstrated predictive importance in this sample.

## 3 · Candidate input features

### Top five initial candidates

1. **Credit_Utilization** — proportion of credit limit used; may capture borrowing pressure, though EDA correlation was only +0.0342.
2. **Debt_to_Income_Ratio** — relative debt burden; segmented EDA showed differing rates but no established significance.
3. **Credit_Score** — summary of creditworthiness; EDA produced an unexpected non-monotonic pattern, requiring validation.
4. **Missed_Payments** — repayment behavior, **conditional on timestamp validation**; EDA correlation was −0.0265.
5. **Account_Tenure** — account history length; EDA correlation was −0.0398.

### Additional candidates

- **Income, Loan_Balance, Age:** evaluate only with clear justification, appropriate data permissions, and governance review.
- **Employment_Status, Credit_Card_Type:** one-hot encode and examine incremental performance and fairness impacts.
- **Loan_Balance_Missing:** retain as an exploratory indicator when appropriate.
- **OnTime_Months_6M, Late_Months_6M, Missed_Months_6M, Payment_Severity_6M:** evaluate as an alternative feature set rather than blindly adding all alongside six monthly variables (redundancy is possible).
- **Location:** **exclude from the initial model** unless a compelling approved business case and fairness review justify its use.
- **Customer_ID:** exclude; it is an identifier, not a legitimate predictive signal.

### Feature-set experiments

- **Set A — Core financial:** top five candidates.
- **Set B — Expanded:** Set A + Income, Loan_Balance, approved categorical features and missingness flag.
- **Set C — Behavioral:** Set B + timestamp-verified payment history features.

Compare feature sets using the same validation splits. Do not interpret improvement on the training set as evidence of predictive value.

## 4 · End-to-end conceptual pipeline

```text
Raw customer record + prediction timestamp
                 │
                 ▼
Validate schema, ranges, units and feature timestamps
                 │
                 ▼
Separate ID / label; split development and untouched test data
                 │
                 ▼
Within training folds ONLY:
  numeric imputation → optional scaling
  categorical imputation/encoding
  missingness flags and approved engineered features
                 │
                 ▼
Train candidate model → predict probabilities
                 │
                 ▼
Cross-validation + hyperparameter tuning
                 │
                 ▼
Calibration, fairness, stability and error analysis
                 │
                 ▼
Lock model and threshold → evaluate ONCE on holdout
                 │
                 ▼
Risk probability + explanation + human review workflow
```

**Recommended data split:** Hold out approximately 20% of records with stratification (about 100 records, potentially only ~16 positive outcomes). On the remaining 80%, use repeated stratified cross-validation, ideally 5 folds where feasible, and report variability. If a real chronological cutoff exists, prefer time-based evaluation over random splits. Because the dataset is small, treat all estimates as uncertain.

**Leakage control:** The Task 1 cleaned dataset is useful for exploration, but for honest modeling, fit imputers, encoders, scaling, and feature selection inside each training fold on original/raw features. Never compute transformations on the full dataset before splitting.

## 5 · Model A: Logistic Regression — recommended baseline

### Mechanism

Logistic Regression models the log-odds of delinquency as an additive function of encoded inputs:

$$\log\!\left(\frac{p}{1-p}\right)=\beta_0+\beta_1 x_1+\cdots+\beta_k x_k,\qquad p=\frac{1}{1+e^{-z}}.$$

A customer’s preprocessed features are multiplied by learned coefficients, summed into a score, and transformed by the logistic function into a probability between 0 and 1. An operating threshold, selected using validation data and business costs, converts that probability into a review flag.

### Why it fits Geldium

- **Explainability:** Coefficients and odds ratios can be inspected; reasons must still be communicated cautiously, particularly with correlated inputs.
- **Data efficiency:** Usually suitable as a starting point for small tabular datasets.
- **Operational simplicity:** Fast to train, score, monitor, and deploy.
- **Governance:** Easier to document and audit than many complex alternatives.

### Constraints and mitigations

- Assumes an additive linear relationship in **log-odds**, not necessarily in raw probability.
- Cannot discover complex interactions without engineered terms; test limited nonlinear transforms only with cross-validation.
- Sensitive to scaling and multicollinearity; use regularization (L2 first; L1 as an optional sparse alternative) and standardized numerical variables.
- Coefficients describe model associations, **not causal effects**.

**Suggested tuning:** Regularization strength `C`, penalty type, class weighting as a comparison (not an automatic default), and probability calibration if needed.

**Go/no-go:** Keep as production candidate only if it beats trivial baselines and demonstrates useful, stable, calibrated out-of-sample performance.

## 6 · Model B: Decision Tree — transparent nonlinear challenger

### Mechanism

A Decision Tree recursively selects feature splits (for example, `Credit_Utilization > threshold`) to partition customers into groups with different delinquency proportions. The terminal leaf supplies an estimated probability based on its training observations.

### Strengths

- Human-readable rules and easy-to-visualize paths.
- Naturally handles nonlinear cutoffs and interactions.
- Often requires little numerical scaling.

### Limitations

- Can overfit badly with **only 80 positive labels**.
- Small data changes may produce very different split rules.
- Deep trees create tiny leaves with unstable probabilities.

### Proposed controls

Limit `max_depth`, increase `min_samples_leaf`, and apply cost-complexity pruning (`ccp_alpha`). Evaluate leaf sample sizes and calibration. A shallow tree may be more useful as an **explanatory comparison** than as the best predictor.

**Interpretation example (illustrative, not learned):** A tree *might* branch on utilization and DTI; the actual split variables and thresholds cannot be known without fitting the model.

## 7 · Model C: Random Forest — robust nonlinear challenger

### Mechanism

Random Forest trains many Decision Trees on bootstrapped training samples, using random subsets of candidate features at splits. It averages their predicted class probabilities to reduce variance relative to a single tree.

### Strengths

- Captures nonlinear patterns and feature interactions without manually specifying them.
- Less sensitive than one deep tree to particular training observations.
- Useful benchmark for mixed financial tabular data.

### Limitations

- Harder to explain globally and per customer than Logistic Regression.
- Can still overfit small samples or learn noise.
- Raw probabilities may be poorly calibrated; impurity-based feature importance can be misleading.

### Proposed controls

Tune `n_estimators`, `max_depth`, `min_samples_leaf`, and `max_features` with stratified cross-validation. Prefer held-out permutation importance and partial dependence/SHAP only after validating model performance and respecting correlated-feature limitations. Check calibration explicitly.

**Selection rule:** Prefer over Logistic Regression only if its incremental benefit is consistent across folds and justified by added complexity.

## 8 · Model D: Gradient Boosting — performance-oriented challenger

### Mechanism

Gradient Boosting builds an ensemble sequentially: each new small tree attempts to correct errors in the current model under a chosen loss function (commonly logistic loss for binary classification). The ensemble produces a score transformed into a delinquency probability.

### Strengths

- Flexible modeling of nonlinearities and interactions.
- Often competitive on structured/tabular prediction tasks.
- Can handle complex relationships with appropriate regularization.

### Limitations

- More hyperparameters and higher overfitting risk on 500 rows.
- Lower intrinsic interpretability than Logistic Regression.
- Calibration, stability, and fairness must be tested rather than assumed.

### Proposed controls

Start with shallow trees, conservative learning rates, and a limited number of boosting stages. Tune tree depth/leaf size, learning rate, and iterations inside cross-validation. Use early stopping only with an internal validation split, never the final test set.

**Selection rule:** Use only if robust cross-validation and untouched-test evidence demonstrate a worthwhile benefit and the governance requirements can be met.

## 9 · Comparison and model recommendation

| Criterion | Logistic Regression | Decision Tree | Random Forest | Gradient Boosting |
|---|---|---|---|---|
| Explainability | **High** | High if shallow | Moderate–low | Moderate–low |
| Nonlinear relationships | Needs engineered terms | Yes | Yes | Yes |
| Small-sample suitability | **Good baseline** | Risk of instability | Requires careful control | Requires careful control |
| Deployment simplicity | **High** | High | Medium | Medium |
| Probability calibration | Must validate | Must validate | Must validate | Must validate |
| Primary role | **Recommended starting model** | Interpretable challenger | Nonlinear challenger | Advanced challenger |

### Chosen model — 2–3 sentence submission summary

**Geldium should initially use regularized Logistic Regression to estimate each customer's probability of delinquency from verified financial and repayment features.** The model will preprocess inputs inside leakage-safe training folds, produce a calibrated risk probability, and support transparent human review using a validation-selected threshold. Decision Tree, Random Forest, and Gradient Boosting will be assessed as challengers, but none should replace the baseline without demonstrated out-of-sample gains.

### One-paragraph justification

Logistic Regression is the preferred initial approach because Geldium needs an interpretable, computationally efficient, and auditable risk-scoring method, especially given the small dataset of 500 customers and 80 delinquent cases. Its regularized coefficients provide a transparent starting point for understanding model behavior, and the method is straightforward to deploy, monitor, and document for financial-services governance. The EDA did not reveal strong standalone predictors, so a more complex model is not automatically warranted; instead, Logistic Regression establishes a defensible benchmark against which nonlinear alternatives can be tested. Any operational use must also include probability calibration, bias checks, and human oversight, with feature timing and intended prediction horizon confirmed before training.

## 10 · Evaluation framework: accuracy, risk and reliability

### 10.1 Validation design

1. Define the target horizon and ensure predictor timestamps precede it.
2. Set aside a stratified untouched holdout; if the task is temporal, use an appropriate out-of-time holdout.
3. On the development set, run repeated stratified cross-validation for preprocessing, tuning, feature-set selection and threshold selection.
4. Compare against **always non-delinquent** and **constant 16%-prevalence probability** baselines.
5. Freeze the entire pipeline and threshold, then evaluate **once** on the holdout.
6. Report confidence intervals/variation, not only point estimates. With ~16 positives expected in a 20% holdout, uncertainty will be substantial.

### 10.2 Metrics and how to interpret them

| Metric | Why it matters | Interpretation / caution |
|---|---|---|
| **Recall (delinquent class)** | How many actual delinquent customers are found? | Higher recall reduces missed cases, often at cost of more false alerts |
| **Precision** | Of flagged customers, how many are delinquent? | Low precision creates avoidable outreach workload |
| **F1** | Balances precision and recall | Useful summary at a chosen threshold, not a substitute for costs |
| **PR-AUC / Average Precision** | Ranking quality with 16% positive prevalence | Especially informative for imbalanced outcomes; prevalence baseline ≈ 0.16 |
| **ROC-AUC** | Ranking discrimination across thresholds | Can appear favorable despite weak precision when positives are rare |
| **Balanced accuracy** | Accounts for both classes | More informative than plain accuracy for imbalance |
| **Confusion matrix** | Counts TP, FP, FN, TN | Connects errors to real operational impact |
| **Brier score / calibration curve** | Are predicted probabilities trustworthy? | A model saying 20% risk should be correct ~20% of the time in comparable groups |
| **Accuracy** | Overall correctness | Majority-only baseline already reaches **84%**, so accuracy alone misleads |

**No metric values are reported here because the models have not been trained.**

### 10.3 Decision threshold and business cost

A 0.50 threshold is not inherently optimal. Compare thresholds on development folds using the cost of missed delinquency versus unnecessary outreach, available reviewer capacity, and the intended intervention. Document the final threshold and obtain business approval before applying it to the holdout.

**Illustrative only:** A probability of 0.27 would be a *27% estimated risk*, not proof the customer will become delinquent. Whether it triggers review depends on a separately selected threshold.

## 11 · Responsible AI: bias, explainability and fairness

> **Status:** Proposed controls, **not** completed audits. The EDA contains no fitted model predictions, so model fairness cannot yet be measured. This section defines how Geldium would audit a model before any operational use.

### 11.1 Distinguish three concepts

| Concept | Practical question | Geldium control |
|---|---|---|
| **Bias** | Could training data, labels, selection, or proxies create systematic error? | Review data provenance, representation, label timing, and subgroup errors. |
| **Explainability** | Can an analyst justify the specific factors behind a risk score? | Transparent baseline, validated local explanations, reason-code review. |
| **Fairness** | Are error rates or adverse outcomes disproportionately distributed across groups? | Predefined group-based tests, uncertainty estimates, escalation and remediation. |

### 11.2 Sources of bias and their relevance

**Historical bias:** Past credit decisions may affect who received loans and therefore whose repayment outcomes were observed. A model trained only on approved customers may inherit this selection effect.

**Selection and sample-size bias:** Geldium's dataset contains **500 records and 80 delinquent outcomes**. Subgroup estimates may be unstable, especially after splitting into training and test sets. Check counts and confidence intervals before drawing conclusions.

**Proxy bias:** Location, employment status, income, and other variables can correlate with legally protected characteristics in a deployment jurisdiction. Such features require documented justification, review, and, where appropriate, exclusion or restrictions.

**Measurement / label bias:** Verify that `Delinquent_Account` is consistently defined and that the six `Month_*` values are known **before** the prediction date. Otherwise, leakage can create misleading performance and group disparities.

**EDA evidence is not proof of bias:** Observed delinquency rates differed by employment status, location, and card type, but the categorical association tests were not statistically significant. These comparisons do **not** establish that a model treats groups unfairly.

### 11.3 Fairness metrics — definitions and interpretation

Let `ŷ = 1` mean the model flags a customer as high risk, `Y = 1` mean actual delinquency, and `A` represent an audited group. Choose the threshold and fairness objectives **before** examining the final test set.

| Metric | Definition | Interpretation / limitation |
|---|---|---|
| **Selection / flag rate** | $P(\hat Y=1\mid A=a)$ | Share flagged in each group; unequal rates alone do not prove unfairness. |
| **Demographic parity difference** | $P(\hat Y=1\mid A=a)-P(\hat Y=1\mid A=b)$ | Compares flag rates, but ignores genuine differences in outcome prevalence. |
| **True positive rate (TPR / recall)** | $TP/(TP+FN)$ | Of actually delinquent customers, proportion correctly flagged. |
| **False positive rate (FPR)** | $FP/(FP+TN)$ | Of non-delinquent customers, proportion incorrectly flagged; especially important when flags cause burdens. |
| **Equal opportunity gap** | $TPR_a-TPR_b$ | Compares recall across groups. |
| **Equalized odds gaps** | Compare both $TPR$ and $FPR$ across groups | Checks error-rate parity; may conflict with calibration and other fairness goals. |
| **Disparate impact ratio** | $P(\hat Y=1\mid A=a)/P(\hat Y=1\mid A=b)$ | Requires careful choice of favorable vs adverse outcome and reference group; a ratio alone is not a legal finding. |
| **Group calibration** | Compare predicted risk with observed outcome frequency within each group | Tests whether a predicted 20% risk corresponds to about 20% observed delinquency, with uncertainty. |

**Important:** A threshold such as the “80% rule” is at most a screening heuristic in certain contexts, not a universal legal test. Legal requirements depend on the jurisdiction and actual use case.

### 11.4 Proposed fairness audit workflow

1. **Define the decision:** Is the score for supportive outreach, manual review, or credit denial? The consequences determine the harm and fairness priorities.
2. **Confirm lawful audit attributes:** Use legally permitted, securely governed protected-group data for auditing where available; do not infer protected characteristics from names or ZIP codes using GenAI.
3. **Check representation:** Report group counts, positive counts, missingness, and overlap. Suppress or combine sparse groups only with documented justification.
4. **Evaluate on untouched data:** Compute group-level confusion matrices, TPR, FPR, precision, calibration, and selection rates, with confidence intervals.
5. **Examine intersectionality:** Where sample sizes permit, check combinations of relevant characteristics; 500 records may be insufficient for reliable intersectional conclusions.
6. **Investigate disparities:** Review proxy features, target construction, sampling, and threshold sensitivity. Test whether patterns persist under resampling.
7. **Mitigate carefully:** Consider data quality fixes, representative data collection, constrained models, feature review, or threshold changes after legal and business review; assess trade-offs and retest on held-out data.
8. **Document and monitor:** Record approvals, model version, thresholds, drift, subgroup metrics, human overrides, and complaints.

**Data limitation:** The reported Geldium variables include age, income, employment, and location, but do not document protected characteristics such as race or sex. A full protected-class fairness audit **cannot be claimed** from the available EDA alone.

### 11.5 Explainability and customer-level reasons

- **Logistic Regression:** Standardized coefficients describe direction and magnitude of changes in *log-odds*, holding other inputs fixed. They are not causal effects; correlated features and encoding choices matter.
- **Decision Tree:** Show the actual sequence of splits leading to a leaf; verify the tree is stable enough to be used for explanations.
- **Random Forest / Gradient Boosting:** Use model-specific or SHAP-based local attributions; compare with global behavior, document the reference dataset, and test explanation stability.
- **Reason codes:** Map validated contributions to plain-language descriptions without asserting unsupported causality. Provide meaningful review and correction routes when decisions affect customers.

**Illustrative only — not a real Geldium prediction:** “The model assigned a higher score primarily because of utilization and debt-to-income values.” Such a statement is acceptable **only if** verified by the fitted model's actual local explanation. Do not ask GenAI to invent the reasons.

### 11.6 Governance and human oversight

For high-impact credit decisions, require documented model purpose, audit logs, explainable adverse-action reasoning where applicable, data security, fairness review, monitoring, and human escalation. This notebook is not a legal compliance determination. A human reviewer must verify GenAI-drafted explanations against actual model outputs and applicable rules.


### 11.7 Proposed responsible-AI acceptance gates

A model **must not be approved solely on overall AUC**. Before any operational use, Geldium should require: (1) a defined decision purpose and prediction horizon, (2) reproducible held-out performance with uncertainty, (3) calibration at relevant risk levels, (4) subgroup fairness testing where lawful attributes and adequate counts exist, (5) validated individual explanations and review procedures, and (6) documented human approval. If protected-group information is unavailable, record the audit as **incomplete**, not “passed.”


## 12 · Model selection decision rules

A candidate model is eligible only if it passes **all** of these checks:

1. **Data validity:** Target definition, timestamps, missingness treatment and feature permissions are documented.
2. **Better than baselines:** Cross-validated PR-AUC, recall/precision trade-off, and calibration provide practical value beyond trivial predictions.
3. **Stable estimates:** Performance is reasonably consistent across folds and bootstrap intervals; unstable gains are not treated as conclusive.
4. **Explainable decisions:** Staff can explain the model’s outputs and limitations in an approved manner.
5. **Fairness and compliance:** No unaddressed material disparity or prohibited proxy use.
6. **Operational fit:** Threshold meets review capacity and business cost constraints.

**Default recommendation:** If no model passes these tests, do **not** deploy. Request more temporally valid, representative labeled data and revisit the problem definition.

## 13 · How GenAI assists in predictive modeling

**Tool used:** ChatGPT (GPT-6) as a drafting and conceptual-scaffolding assistant. **Human responsibility:** Verify all data claims, implementation details, statistics, and regulatory interpretations. GenAI suggestions are not evidence that a model was trained or audited.

### 13.1 Selecting the right model type

GenAI can compare candidate algorithms against the task's constraints. The analyst must make the final selection based on the target, sample size, class imbalance, explainability needs, and observed validation results.

| Model | What GenAI helps outline | Analyst's judgment for Geldium |
|---|---|---|
| **Logistic Regression** | Binary probability, regularization, coefficient interpretation | **Recommended initial baseline**: transparent and practical with 500 observations; test whether performance is adequate. |
| **Decision Tree** | Interpretable splits and depth/pruning choices | Useful nonlinear comparison, but small samples can yield unstable rules. |
| **Random Forest** | Ensemble logic, feature interactions, tuning | Strong nonlinear challenger; harder to explain individual decisions. |
| **Gradient Boosting** | Sequential corrections and calibration considerations | Performance challenger; more tuning and governance overhead. |
| **Neural Network** | Layer architecture and nonlinear decision surfaces | Not a first choice for only 500 records without demonstrated benefit; greater overfitting and explanation concerns. |

**Example GenAI prompt:**

> “Given a credit delinquency dataset with 500 customers, 80 positive cases, weak univariate associations, categorical payment histories, and financial-service explainability requirements, compare Logistic Regression, Decision Tree, Random Forest, Gradient Boosting, and a Neural Network. Recommend a baseline and challengers without inventing performance scores.”

**Human review:** Confirm the recommendation does not confuse correlations with validated importance or claim a model is “best” before testing.

### 13.2 Generating model code or pseudocode

GenAI can scaffold Python, R, or SQL workflows, but generated code requires manual inspection and execution tests. For this assignment, the pipeline is conceptual; **no model is trained here**.

**Example GenAI prompt:**

> “Draft *pseudocode* for a scikit-learn credit delinquency classifier using a stratified holdout, training-only imputation, one-hot encoding, regularized Logistic Regression, probability calibration where needed, threshold selection on validation data, and final test evaluation. Prevent target and temporal leakage. Explain each step.”

**Reviewed pseudocode (not executable training code):**

```text
1. Define outcome window and verify all features predate the outcome.
2. Remove Customer_ID from predictors; separate X and y.
3. Create a stratified untouched test set; reserve validation folds within training.
4. In each training fold: fit numeric imputer, categorical encoder and model.
5. Compare regularized Logistic Regression with tree-based challengers.
6. Use validation predictions to assess PR-AUC, ROC-AUC and calibration.
7. Select an intervention threshold using documented business costs.
8. Audit subgroup errors, proxies, uncertainty and local explanations.
9. Freeze the chosen pipeline; evaluate once on untouched test data.
10. Document intended use, monitoring, review and retraining triggers.
```

**Review checklist:** Correct feature timing; no imputation fitted before splitting; no ID leakage; no fabricated metrics; reproducible seeds; sensible treatment of class imbalance; proper threshold and calibration evaluation; privacy and fairness review.

### 13.3 Evaluating performance with GenAI support

GenAI can explain the confusion matrix and help interpret measured precision, recall, F1, ROC-AUC, PR-AUC, Brier score, calibration, and subgroup metrics. It **cannot evaluate performance without actual predictions and outcomes**.

**Example GenAI prompt:**

> “Using the attached *measured* validation confusion matrix and subgroup counts, calculate precision, recall, F1, false-positive rates, and uncertainty intervals. Compare performance against a no-skill baseline of 16% delinquency prevalence. Identify possible fairness concerns, distinguish findings from hypotheses, and do not infer protected-group fairness where attributes are unavailable.”

**How to interpret results:**

- **High accuracy alone is insufficient:** Predicting everyone as non-delinquent yields **84% accuracy** and **zero recall** for delinquency in this dataset.
- **PR-AUC:** Especially informative with 16% positives; compare to a no-skill prevalence baseline of approximately 0.16 under the same evaluation conditions.
- **Recall vs precision:** A low threshold may find more at-risk customers but can increase false alarms; document costs and capacity.
- **Calibration:** Check whether predicted probabilities match observed event rates before using scores as risk estimates.
- **Fairness:** Compare relevant group error rates and uncertainty; GenAI cannot certify legal or ethical fairness.

### 13.4 Prompt → draft → verification → decision

| Stage | GenAI contribution | Required analyst action |
|---|---|---|
| Model selection | Explains trade-offs and proposes candidates | Confirm suitability for Geldium and justify baseline. |
| Workflow generation | Produces pseudocode and preprocessing ideas | Check leakage, correctness, data types and timing. |
| Performance analysis | Helps explain *supplied* measured metrics | Recompute metrics from actual predictions and inspect uncertainty. |
| Explainability | Helps translate verified SHAP/coefficient outputs | Check each reason against the fitted model; reject invented explanations. |
| Fairness | Suggests metrics and audit design | Run formal tests, assess group sizes, obtain domain/legal review. |

### 13.5 Transparent GenAI use statement

> “ChatGPT was used to structure the modeling plan, compare candidate algorithms, draft conceptual pseudocode, and propose evaluation, explainability, and fairness checks. The Geldium dataset statistics are drawn from the prior EDA. No models were trained and no predictive accuracy or fairness results are claimed in this conceptual notebook. All generated suggestions require independent validation before implementation.”


In [ ]:
# Optional: inspect the evidence table used in this conceptual report.
# This is NOT model training and does not produce predictive performance.
eda_facts = {
    "customers": 500,
    "non_delinquent": 420,
    "delinquent": 80,
    "delinquency_prevalence": 80 / 500,
    "majority_class_accuracy_baseline": 420 / 500,
}
for name, value in eda_facts.items():
    print(f"{name}: {value:.2%}" if isinstance(value, float) else f"{name}: {value}")

### Scope and source note

Dataset values and reported statistical outcomes in this notebook come from the user's prior Geldium EDA report. Model descriptions, evaluation methods, and governance recommendations are **proposed** conceptual guidance. No raw dataset was supplied for independent verification in this deliverable, and no predictive model was trained.